In [0]:
df = spark.table("testdatabricksuni.uniqueschema.bank_customer_stage2")

In [0]:
df_clean = df.dropDuplicates(["customer_id"])

print(df.count())
print(df_clean.count())

103000
100000


In [0]:
from pyspark.sql.functions import trim
from pyspark.sql.functions import col
for c in df_clean.columns:
    df_clean = df_clean.withColumn(c, trim(col(c)))

In [0]:
from pyspark.sql.functions import lower, when

df_clean = df_clean.withColumn(
    "gender",
    when(lower(col("gender")).isin("male", "m"), "Male")
    .when(lower(col("gender")).isin("female", "f"), "Female")
    .otherwise("Unknown")
)

In [0]:
from pyspark.sql.types import DoubleType

df_clean = df_clean.withColumn("income", col("income").cast(DoubleType()))
median_income = df_clean.approxQuantile("income", [0.5], 0.01)[0]

df_clean = df_clean.fillna({
    "income": int(median_income)
})

In [0]:
df_clean = df_clean.fillna({
    "state": "Unknown"
})

In [0]:
from pyspark.sql.functions import when

df_clean = df_clean.withColumn(
    "state",
    when(col("state")=="MH","Maharashtra")
    .when(col("state")=="Gujrat","Gujarat")
    .when(col("state")=="Karnatak","Karnataka")
    .when(lower(col("state"))=="delhi","Delhi")
    .when(col("state")=="TN","Tamil Nadu")
    .otherwise(col("state"))
)

In [0]:
from pyspark.sql.functions import regexp_extract

email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

df_clean = df_clean.withColumn(
    "email_valid",
    col("email").rlike(email_regex)
)

# Display Invalid mail id's
display(
    df_clean.filter(~col("email_valid"))
)

# Display Valid mail id's
display(
    df_clean.filter(col("email_valid"))
)

In [0]:
from pyspark.sql.functions import regexp_replace

df_clean = df_clean.withColumn(
    "phone",
    regexp_replace(col("phone"), "[^0-9]", "")
)

# Create a phone validity flag:

df_clean = df_clean.withColumn(
    "phone_valid",
    (col("phone").rlike("^[0-9]{10}$"))
)

display(df_clean)

In [0]:
df_clean.write.mode("overwrite").saveAsTable(
    "testdatabricksuni.uniqueschema.bank_customer_stage3"
)